# Load seeds: bronze seed tables

Loads the CSV files in `seed/` as received into `bronze.raw_dim_market`, `bronze.raw_dim_commodity` and
`bronze.raw_dim_date` in the workspace's default catalog (the tables are created by `src/setup`).

- CSV files only. The column names must match the table's columns, in the same order. Values stay strings; typing and
  data quality checks happen in `src/load_silver_dims`.
- Append only: every run adds the file's rows with `_run_id` set to the `run_id` widget, so a bad file stays visible in bronze.
- An empty file is refused.

Parameters:
- `run_id` (required): id of the run, written to `_run_id`.
- `seed_dir` (optional): folder holding the CSVs. Default: the `seed` folder next to `src`.
- `fail_on_error` (optional, `true`/`false`): raise at the end if a table was not loaded, so a job task fails.

If a table fails to load, the reason is printed and the notebook still completes unless `fail_on_error` is `true`.

In [ ]:
dbutils.widgets.text("run_id", "", "Run id")
dbutils.widgets.text("seed_dir", "", "Seed folder (optional)")
dbutils.widgets.text("fail_on_error", "false", "Fail the job on error (true/false)")

## Functions

In [ ]:
def get__seed_dir(seed_dir_param: str) -> str:
    """Return the folder holding the seed CSVs.

    Args:
        seed_dir_param: Value of the seed_dir widget; empty to use the default.

    Returns:
        The given folder, or the "seed" folder next to "src" in the workspace
        (this notebook is src/load_bronze_dims); empty string if it could not be worked out
        (the reason is printed).
    """
    import os

    if seed_dir_param:
        return seed_dir_param
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        repo_root = os.path.dirname(os.path.dirname(notebook_path))
        return f"/Workspace{repo_root}/seed"
    except Exception as e:
        print(f"ERROR: could not work out the seed folder, pass the seed_dir widget: {e}")
        return ""

## Run

In [ ]:
import os

from lib_mandi.config import OPTIONS_READ_SEED, SCHEMA_RAW_DIM_COMMODITY, SCHEMA_RAW_DIM_DATE, SCHEMA_RAW_DIM_MARKET
from lib_mandi.extract import extract__seed_file
from lib_mandi.load import load__bronze
from lib_mandi.transformations import transform__bronze
from lib_mandi.utility import get__default_catalog

run_id = dbutils.widgets.get("run_id").strip()
if not run_id:
    raise ValueError("Widget 'run_id' is required")

catalog = get__default_catalog(spark)
print(f"Loading seed files into bronze in catalog: {catalog}")

if not catalog:
    raise RuntimeError("Could not read the default catalog (see the error printed above)")

seed_dir = get__seed_dir(dbutils.widgets.get("seed_dir").strip())
if not seed_dir:
    raise RuntimeError("Could not work out the seed folder (see the error printed above)")
print(f"Seed folder: {seed_dir}")

seeds = [
    ("dim_market.csv", f"{catalog}.bronze.raw_dim_market", SCHEMA_RAW_DIM_MARKET),
    ("dim_commodity.csv", f"{catalog}.bronze.raw_dim_commodity", SCHEMA_RAW_DIM_COMMODITY),
    ("dim_date.csv", f"{catalog}.bronze.raw_dim_date", SCHEMA_RAW_DIM_DATE),
]

loaded = True
for file_name, table_name, schema in seeds:
    df = extract__seed_file(spark, f"{seed_dir}/{file_name}", schema, OPTIONS_READ_SEED)
    if df is not None:
        df = transform__bronze(df, run_id, os.path.basename(file_name))
    if df is not None:
        loaded = load__bronze(df, table_name) and loaded
    else:
        loaded = False

if not loaded and dbutils.widgets.get("fail_on_error").strip().lower() == "true":
    raise RuntimeError("Load did not finish (see the reason printed above)")